In [0]:
#%run ./dim_language ----- A décommmenter pour lancer les notebooks séparements

# build_translations

Construit les tables de traduction du modèle à partir des tables
`*_translations` de la base PostgreSQL.

## Pourquoi une transformation est nécessaire

Les tables source ne peuvent pas être branchées telles quelles sur le modèle :

1. **Le fallback demandé par le PO** (langue -> anglais -> la clé, jamais de null)
   n'est pas réalisable en RLS. Le RLS sait seulement *supprimer* des lignes, pas
   en substituer une autre. Le repli doit donc être matérialisé ici.
2. **Les traductions sont incomplètes** : toutes les langues ne sont pas
   renseignées pour toutes les clés. Sans traitement, une clé non traduite
   *disparaîtrait* du visuel après filtrage RLS — les lignes seraient absentes et
   **les totaux seraient faux**, ce qui est bien plus grave qu'un libellé anglais.
3. **Le couple (clé, langue) doit être unique et toujours présent** pour qu'après
   filtrage RLS il reste exactement une ligne par clé. Sinon les visuels
   dupliquent les lignes.

On produit donc des tables **denses** : le produit cartésien
`clés x langues actives`, avec un libellé garanti non nul sur chaque ligne.

In [0]:
# Langue de repli (English = 2 dans parameters_languages)
LANG_EN = 2


def build_translation_dim(df_trad, key_cols, label_col="label", fallback_expr=None):
    """Rend dense une table *_translations source.

    Retourne une ligne par (clé x langue active), avec :
      - label        : libellé garanti non nul (langue -> anglais -> clé)
      - label_source : d'où vient le libellé, pour le contrôle qualité
    """
    actives = df_trad.filter(F.col("deleted") == False)

    # 1. Dédoublonnage : une seule ligne par (clé, langue), la plus récente.
    #    Sans cela un doublon en base dupliquerait les lignes dans tous les visuels.
    w = Window.partitionBy(*key_cols, "language").orderBy(
        F.coalesce(F.col("updated_at"), F.col("created_at")).desc_nulls_last()
    )
    actives = (
        actives
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # 2. Grille dense : toutes les clés x toutes les langues ACTIVES du référentiel.
    #    Le produit cartésien se fait sur dim_language, pas sur les langues
    #    présentes dans la table source : c'est ce qui garantit qu'une langue non
    #    traduite obtient malgré tout une ligne (avec repli) au lieu de disparaître.
    keys = actives.select(*key_cols).distinct()
    grid = keys.crossJoin(F.broadcast(dim_language.select("language")))

    # 3. Libellé anglais, support du repli de niveau 2
    label_en = (
        actives.filter(F.col("language") == LANG_EN)
               .select(*key_cols, F.col(label_col).alias("_label_en"))
    )

    # 4. Repli de niveau 3 : la clé elle-même, jamais de null
    if fallback_expr is None:
        fallback_expr = F.concat_ws("_", *[F.col(c).cast("string") for c in key_cols])

    return (
        grid
        .join(
            actives.select(*key_cols, "language", F.col(label_col).alias("_label")),
            key_cols + ["language"], "left"
        )
        .join(label_en, key_cols, "left")
        .withColumn("label", F.coalesce(F.col("_label"), F.col("_label_en"), fallback_expr))
        .withColumn(
            "label_source",
            F.when(F.col("_label").isNotNull(), F.lit("translated"))
             .when(F.col("_label_en").isNotNull(), F.lit("fallback_en"))
             .otherwise(F.lit("fallback_key"))
        )
        .select(*key_cols, "language", "label", "label_source")
    )

In [0]:
def ensure_delta_table(df, table_name):
    """Crée la table Delta cible si elle n'existe pas encore (schéma seul, 0 ligne).

    handle_table_update ne sait pas créer une table : DeltaTable.forName échoue si
    elle est absente. Ce bootstrap rend le notebook rejouable sur un environnement
    neuf (dev, preprd) sans création manuelle préalable.

    La table est créée avec le schéma du DataFrame plus created_at, colonne que
    handle_table_update ajoute systématiquement. Pas d'updated_at : en mode "full"
    la fonction fait delete() puis write.mode("append"), et un append Delta exige
    des schémas qui correspondent exactement.
    """
    if not spark.catalog.tableExists(table_name):
        (
            df.limit(0)
              .withColumn("created_at", F.current_timestamp())
              .write.format("delta")
              .saveAsTable(table_name)
        )
        print(f"Table {table_name} créée (schéma seul, 0 ligne).")


# Noms des tables effectivement écrites, alimenté par publish_translation_dim.
# Sert au contrôle de couverture en fin de notebook, qui relit les tables au lieu
# de recalculer les DataFrames.
tables_publiees = []


def publish_translation_dim(df, process_name, primary_key):
    """Écrit une table de traduction avec la mécanique Delta du projet.

    mode="full" est imposé ici, et non execution_mode. En mode "update",
    handle_table_update ne fait qu'insérer et mettre à jour : il ne supprime
    jamais. Une clé retirée à la source resterait donc indéfiniment dans la table
    du modèle et continuerait d'apparaître dans les slicers du rapport — un
    libellé fantôme, sans données derrière. Ces tables sont petites et
    entièrement redérivées à chaque run : le remplacement complet est à la fois
    plus sûr et sans coût notable.
    """
    target = current_catalog + "." + current_schema + "." + process_name
    all_columns = df.columns
    additional_columns = get_additional_columns(all_columns, primary_key)

    if verbose_mode == 'debug':
        print(f"{target} -> clé {primary_key}, colonnes {all_columns}")

    ensure_delta_table(df, target)

    handle_table_update(
        df,
        target,
        primary_key,
        all_columns,
        additional_columns_to_check=additional_columns,
        mode="full"
    )

    tables_publiees.append((process_name, target))
    return target

## Traductions à clé simple

Ces cinq tables partagent le même patron : une clé entière, une langue, un
libellé. La clé est renommée pour correspondre à la colonne portée par la table
du modèle qui s'y rattachera.

In [0]:
# goods_species_translations -> dim_trad_specy
dim_trad_specy = build_translation_dim(
    goods_species_translations,
    key_cols=["good_specy"]
).withColumnRenamed("good_specy", "id_good_specy")

publish_translation_dim(dim_trad_specy, "dim_trad_specy", ["id_good_specy", "language"])

In [0]:
# goods_varieties_translations -> dim_trad_variety
dim_trad_variety = build_translation_dim(
    goods_varieties_translations,
    key_cols=["good_variety"]
).withColumnRenamed("good_variety", "id_good_variety")

publish_translation_dim(dim_trad_variety, "dim_trad_variety", ["id_good_variety", "language"])

In [0]:
# parameters_production_type_translations -> dim_trad_production_type
# La source nomme sa clé "id_parameters_production_type" (pluriel) alors que la
# table métier parameters_production_types porte "id_parameter_production_type"
# (singulier). On normalise ici sur le nom singulier, celui du modèle.
dim_trad_production_type = build_translation_dim(
    parameters_production_type_translations,
    key_cols=["id_parameters_production_type"]
).withColumnRenamed("id_parameters_production_type", "id_parameter_production_type")

publish_translation_dim(
    dim_trad_production_type, "dim_trad_production_type",
    ["id_parameter_production_type", "language"]
)

In [0]:
# parameters_variables_translations -> dim_trad_variable
dim_trad_variable = build_translation_dim(
    parameters_variables_translations,
    key_cols=["parameter_variable"]
).withColumnRenamed("parameter_variable", "id_parameter_variable")

publish_translation_dim(
    dim_trad_variable, "dim_trad_variable", ["id_parameter_variable", "language"]
)

In [0]:
# parameters_production_line_variables_translations -> dim_trad_production_line_variable
dim_trad_production_line_variable = build_translation_dim(
    parameters_production_line_variables_translations,
    key_cols=["parameter_production_line_variable"]
).withColumnRenamed(
    "parameter_production_line_variable", "id_parameter_production_line_variable"
)

publish_translation_dim(
    dim_trad_production_line_variable, "dim_trad_production_line_variable",
    ["id_parameter_production_line_variable", "language"]
)

In [0]:
# parameters_localizations_translations -> dim_trad_localization
dim_trad_localization = build_translation_dim(
    parameters_localizations_translations,
    key_cols=["id_parameter_localization"]
)

publish_translation_dim(
    dim_trad_localization, "dim_trad_localization",
    ["id_parameter_localization", "language"]
)

## Traduction à clé composite : les groupes de localisation

`parameters_localization_groups_translations` est la seule table dont la clé
porte deux colonnes : un même groupe de localisation a un libellé différent
selon la ligne de production. La clé du modèle est donc
(`id_parameter_localization_group`, `production_line`).

In [0]:
dim_trad_localization_group = build_translation_dim(
    parameters_localization_groups_translations,
    key_cols=["id_parameter_localization_group", "production_line"]
)

publish_translation_dim(
    dim_trad_localization_group, "dim_trad_localization_group",
    ["id_parameter_localization_group", "production_line", "language"]
)

## Traductions des notes de production

`parameters_batch_note_categories_translations` sert **quatre usages** dans une
seule table : la clé `batch_note_category` est préfixée par le type
(`location_...`, `event_...`, `detail_...`, `impact_...`).

On en produit **quatre tables distinctes** plutôt qu'une seule. Raison :
`fact_batch_note` porte quatre colonnes à traduire (location, event, detail,
impact), et Power BI n'autorise qu'**une seule relation active** entre deux
tables. Une table unique obligerait à trois relations inactives et à des
`USERELATIONSHIP` dans chaque mesure — ingérable sur des colonnes posées
directement sur les axes des visuels.

Le repli de niveau 3 affiche le code **sans son préfixe** (`TCR` et non
`detail_TCR`) : c'est ce que l'utilisateur reconnaît.

In [0]:
# Le préfixe est séparé du code par le premier "_" seulement :
# "detail_defaut_niveau_haut_BOM3" -> type "detail", code "defaut_niveau_haut_BOM3"
batch_note_trad_base = (
    parameters_batch_note_categories_translations
    .withColumn("category_type", F.split(F.col("batch_note_category"), "_", 2).getItem(0))
    .withColumn("category_code", F.split(F.col("batch_note_category"), "_", 2).getItem(1))
)

if verbose_mode == 'debug':
    print("Préfixes rencontrés dans batch_note_category :")
    display(
        batch_note_trad_base.filter(F.col("deleted") == False)
                            .groupBy("category_type").count().orderBy(F.desc("count"))
    )

In [0]:
BATCH_NOTE_TYPES = ["location", "event", "detail", "impact"]

for category_type in BATCH_NOTE_TYPES:
    df_type = batch_note_trad_base.filter(F.col("category_type") == category_type)

    dim_type = build_translation_dim(
        df_type,
        key_cols=["batch_note_category"],
        # repli affiché : le code sans son préfixe
        fallback_expr=F.split(F.col("batch_note_category"), "_", 2).getItem(1)
    )

    process_name = f"dim_trad_batch_note_{category_type}"
    globals()[process_name] = dim_type
    publish_translation_dim(dim_type, process_name, ["batch_note_category", "language"])

## Contrôle qualité du rafraîchissement

`label_source` mesure la couverture réelle des traductions. Un taux élevé de
`fallback_key` sur une langue signale que le front n'a pas alimenté la table :
le rapport reste fonctionnel mais s'affiche en anglais ou en codes techniques.
C'est l'indicateur à remonter au PO, pas un incident technique.

In [0]:
# Le contrôle relit les tables Delta qui viennent d'être écrites, il ne rejoue pas
# les DataFrames. Deux raisons :
#   - performance : les DataFrames ne sont pas en cache, les réutiliser ferait
#     recalculer toute la chaîne depuis PostgreSQL (dédoublonnage, crossJoin,
#     jointures) et doublerait le temps du notebook ;
#   - fiabilité : on mesure ce qui est réellement en base, pas ce qui aurait dû
#     y être écrit.
couverture = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .groupBy("language", "label_source")
             .count()
             .withColumn("table", F.lit(nom))
        for nom, target in tables_publiees
    ]
)

display(
    couverture.alias("c")
              .join(F.broadcast(dim_language).alias("l"), "language", "left")
              .groupBy("table", "code")
              .pivot("label_source", ["translated", "fallback_en", "fallback_key"])
              .agg(F.sum("count"))
              .orderBy("table", "code")
)